# 02 — NBA pregame microstructure

The two calibration curves that drive everything: marketable-flow arrival rate R(t) and bounded vol c(t) as functions of hours-to-tip, plus the maker-fee economics.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt

SPORT, SEASON = 'nba', '2024-25'
calib = pathlib.Path('..') / 'data' / 'calib'
intensity = pd.DataFrame(json.loads((calib / f'{SPORT}_{SEASON}_intensity.json').read_text()))
vol = pd.DataFrame(json.loads((calib / f'{SPORT}_{SEASON}_vol.json').read_text()))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].step(intensity['max_h'], intensity['R_per_min'], where='pre'); axes[0].set_xscale('log')
axes[0].set_xlabel('hours to tip'); axes[0].set_ylabel('R (trades/min/side)'); axes[0].set_title('Flow concentrates into tip')
axes[1].step(vol['max_h'], vol['c_per_sqrt_min'], where='pre'); axes[1].set_xscale('log')
axes[1].set_xlabel('hours to tip'); axes[1].set_ylabel('c(t) cents/sqrt-min at mid=50'); axes[1].set_title('Vol ramps into tip (news risk)')
fig.tight_layout()

In [ ]:
# Maker-fee economics: round-trip cost vs the 1c spread, by price level
import numpy as np
from kalshi_mm.sim.fees import FeeSchedule
fees = FeeSchedule()
p = np.arange(2, 99)
rt_fee = np.array([fees.maker_fee_c(x, 100) + fees.maker_fee_c(x+1, 100) for x in p]) / 100
plt.figure(figsize=(8,4))
plt.plot(p, rt_fee, label='maker fees, 100-lot round trip (c/contract)')
plt.axhline(1.0, color='r', ls='--', label='1c spread captured')
plt.xlabel('price (c)'); plt.ylabel('cents per contract'); plt.legend()
plt.title('Fees eat the 1c spread at mid prices — MM is only viable near the wings or with selectivity');

In [ ]:
# Depth-from-mid distribution of tape trades (frac at touch -> fill model design)
from kalshi_mm.data.build import iter_games
from kalshi_mm.calib.intensity import collect_trade_depths
games = []
for i, g in enumerate(iter_games('../data/raw', SPORT, SEASON, pregame_hours=6)):
    games.append(g)
    if i >= 150: break
d = collect_trade_depths(games)
ax = d['delta_c'].clip(upper=5).plot.hist(bins=40, figsize=(8,3), logy=True)
ax.set_xlabel('trade depth from mid (c)'); ax.set_title(f'{(d.delta_c<=0.6).mean():.0%} of pregame trades print at the touch');